# Power Track Frame Analysis with Pyshark

This notebook demonstrates how to analyze captured tick data and extract Power Track frames using pyshark.


In [ ]:
import pyshark
import numpy as np
import pandas as pd
from pathlib import Path
import struct
from typing import List, Tuple


## CRC-7 Calculation


In [ ]:
def crc7(data: bytes, polynomial: int = 0x09) -> int:
    """Calculate CRC-7 checksum."""
    crc = 0
    for byte in data:
        crc ^= byte
        for _ in range(8):
            if crc & 0x80:
                crc = ((crc << 1) ^ polynomial) & 0xFF
            else:
                crc = (crc << 1) & 0xFF
    return crc & 0x7F


## Frame Validation


In [ ]:
def validate_frame(frame_bytes: bytes, xor_mask: int = 0x00):
    """Validate a Power Track frame."""
    if len(frame_bytes) < 7:
        return False, {}
    
    # Apply XOR mask
    unmasked = bytes(b ^ xor_mask for b in frame_bytes[:7])
    
    # Extract header (bytes 0-5)
    header = unmasked[:6]
    
    # Compute CRC-7
    computed_crc = crc7(header)
    
    # Extract expected CRC from trailer
    trailer_byte = unmasked[6]
    expected_crc = (trailer_byte >> 1) & 0x7F
    stop_bit = trailer_byte & 0x01
    
    # Parse fields
    byte0 = unmasked[0]
    opcode = (byte0 >> 2) & 0x3F
    version = byte0 & 0x03
    
    start_time = unmasked[1] | (unmasked[2] << 8)
    
    byte3 = unmasked[3]
    duration_scale = (byte3 >> 2) & 0x3F
    compression_ratio = byte3 & 0x03
    
    anchor_price = unmasked[4]
    
    byte5 = unmasked[5]
    volume_code = (byte5 >> 2) & 0x3F
    parity = byte5 & 0x03
    
    is_valid = (computed_crc == expected_crc) and (stop_bit == 1)
    
    frame_info = {
        'valid': is_valid,
        'opcode': opcode,
        'version': version,
        'start_time_us': start_time,
        'duration_scale': duration_scale,
        'compression_ratio': compression_ratio,
        'anchor_price': anchor_price,
        'volume_code': volume_code,
        'parity': parity,
        'crc7_computed': computed_crc,
        'crc7_expected': expected_crc,
        'stop_bit': stop_bit
    }
    
    return is_valid, frame_info
